Die benötigten Bibliotheken werden importiert

In [ ]:
from random import seed, randint
seed(23)

import simpy

Die Klasse ElektronischesFahrzeug wird definiert.
Der Prozess batterie_steuerung() wird in beim parken initiiert. Anschließend wird der yield Befehl aufgerufen, bis charging oder parking erfüllt ist.
Nach 60 Minuten ist parking vorbei und wenn charging noch nicht fertig ist, wird der Prozess unterbrochen.

In [ ]:
class ElektronischesFahrzeug:
    def __init__(self, env: simpy.Environment):
        self.env = env
        self.drive_proc = env.process(self.drive(env))

    def drive(self, env: simpy.Environment):
        """Das Fahrzeug fährt 20-40 min und startet danach das Parken für 1 h. 
        Dabei wird die Methode batterie_steuerung zum Laden der Batterie aufgerufen. Da diese länger dauert, als das Parken,
         muss das Laden unterbrochen werden."""
        while True:
                # Fahre für 20-40 min
                yield env.timeout(randint(20,40))

                # Parke für 1 h
                print("Starte das parken um: ", env.now)
                charging = env.process(self.batterie_steuerung(env))
                parking = env.timeout(60)
                yield charging | parking
                if not charging.triggered:
                    # Unterbreche charging, wenn dies nicht schon getan wurde
                    charging.interrupt("Muss jetzt los")
                print("Parken endet um:", env.now)   

    def batterie_steuerung(self, env: simpy.Environment):
        print("Batterie laden startet um:", env.now)
        try: 
             yield env.timeout(randint(60, 90))
        except simpy.Interrupt as i:
            # ohh nein! der Prozess wurde unterbrochen, bevor der Ladevorgang abgeschlossen war.
            print("Batterie Steuerung wurde unterbrochen um", env.now, "msg:", i.cause)

env = simpy.Environment()

ev = ElektronischesFahrzeug(env)

env.run(until=100)

